In [ ]:
import marimo as mo

In [ ]:
import sys
import os
import subprocess

# Do not import torch here. The sandbox dependency environment must be
# established first; the H2/ES1 runner imports and validates torch itself.

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"

from pathlib import Path as _HfPath
import zipfile as _hf_zipfile
from huggingface_hub import hf_hub_download

# HF_TOKEN is a runtime secret; never embed a personal access token in notebook source.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
os.environ["HF_TOKEN"] = HF_TOKEN

TINYPERSON_HF_REPO = "duyle2408/TinyPerson"
TINYPERSON_HF_FILENAME = "TinyPerson.zip"
tinyperson_download_dir = _HfPath("/marimo/tinyperson_download")
tinyperson_download_dir.mkdir(parents=True, exist_ok=True)

_expected_archive = tinyperson_download_dir / TINYPERSON_HF_FILENAME
try:
    tinyperson_archive_path = _HfPath(
        hf_hub_download(
            repo_id=TINYPERSON_HF_REPO,
            filename=TINYPERSON_HF_FILENAME,
            repo_type="model",
            token=HF_TOKEN,
            local_dir=str(tinyperson_download_dir),
            force_download=False,
        )
    ).resolve()
except Exception:
    # hf_hub_download may fail on a transient metadata/network request even
    # when a previously completed local archive is already usable.
    if _expected_archive.is_file() and _hf_zipfile.is_zipfile(_expected_archive):
        tinyperson_archive_path = _expected_archive.resolve()
        print("Hugging Face metadata request failed; reusing verified local ZIP.")
    else:
        raise

if not tinyperson_archive_path.is_file():
    raise FileNotFoundError(tinyperson_archive_path)
if not _hf_zipfile.is_zipfile(tinyperson_archive_path):
    raise ValueError(f"Downloaded file is not a valid ZIP: {tinyperson_archive_path}")

print(
    "TinyPerson ZIP ready:",
    tinyperson_archive_path,
    f"({tinyperson_archive_path.stat().st_size / 1024**3:.2f} GiB)",
)

In [ ]:
# ================================================================
# EXTRACT THE COMPLETE TINYPERSON ARCHIVE — run after HF download.
# Resolves the real dataset root regardless of the ZIP's top folder.
# ================================================================
from pathlib import Path as _TpPath, PurePosixPath as _TpPurePosixPath
import json as _tp_json
import zipfile as _tp_zipfile

TINYPERSON_TRAIN_CORNER_REL = _TpPath(
    "erase_with_uncertain_dataset/annotations/corner/task/"
    "tiny_set_train_sw640_sh512_all.json"
)
TINYPERSON_TEST_CORNER_REL = _TpPath(
    "annotations/corner/task/tiny_set_test_sw640_sh512_all.json"
)
TINYPERSON_TEST_MERGED_REL = _TpPath(
    "annotations/task/tiny_set_test_all.json"
)
tinyperson_extract_dir = _TpPath("/marimo/tinyperson_extracted").resolve()


def _resolve_tinyperson_root(_extract_dir: _TpPath) -> _TpPath:
    _matches = list(_extract_dir.rglob(TINYPERSON_TRAIN_CORNER_REL.name))
    _valid_roots = []
    for _match in _matches:
        # .../<root>/erase_with_uncertain_dataset/annotations/corner/task/file.json
        if len(_match.parents) < 5:
            continue
        _root = _match.parents[4]
        if (
            (_root / TINYPERSON_TRAIN_CORNER_REL).is_file()
            and (_root / TINYPERSON_TEST_CORNER_REL).is_file()
            and (_root / TINYPERSON_TEST_MERGED_REL).is_file()
        ):
            _valid_roots.append(_root.resolve())
    _valid_roots = sorted(set(_valid_roots))
    if len(_valid_roots) != 1:
        raise RuntimeError(
            "Expected exactly one extracted TinyPerson dataset root, found: "
            f"{[str(_root) for _root in _valid_roots]}"
        )
    return _valid_roots[0]


if not _tp_zipfile.is_zipfile(tinyperson_archive_path):
    raise ValueError(f"Invalid TinyPerson ZIP: {tinyperson_archive_path}")

tinyperson_extract_dir.mkdir(parents=True, exist_ok=True)
_extract_marker = tinyperson_extract_dir / ".tinyperson_extract_complete.json"
_archive_stat = tinyperson_archive_path.stat()
_archive_signature = {
    "archive": str(tinyperson_archive_path),
    "size": int(_archive_stat.st_size),
    "mtime_ns": int(_archive_stat.st_mtime_ns),
}
_marker_matches = False
if _extract_marker.is_file():
    try:
        _marker_matches = (
            _tp_json.loads(_extract_marker.read_text(encoding="utf-8"))
            == _archive_signature
        )
    except (OSError, ValueError):
        _marker_matches = False

if _marker_matches:
    print("SKIP TinyPerson extraction: completed archive signature matches.")
else:
    print("Extracting the complete TinyPerson archive...")
    with _tp_zipfile.ZipFile(tinyperson_archive_path) as _archive:
        _unsafe_members = []
        for _member in _archive.infolist():
            _member_path = _TpPurePosixPath(_member.filename)
            if _member_path.is_absolute() or ".." in _member_path.parts:
                _unsafe_members.append(_member.filename)
        if _unsafe_members:
            raise ValueError(
                "Refusing unsafe paths in TinyPerson ZIP: "
                f"{_unsafe_members[:5]}"
            )
        _archive.extractall(tinyperson_extract_dir)
    _extract_marker.write_text(
        _tp_json.dumps(_archive_signature, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print("TinyPerson archive extraction complete.")

tinyperson_data_root = _resolve_tinyperson_root(tinyperson_extract_dir)
print("Resolved TinyPerson data root:", tinyperson_data_root)
print("Train corner annotations:", tinyperson_data_root / TINYPERSON_TRAIN_CORNER_REL)
print("Test corner annotations:", tinyperson_data_root / TINYPERSON_TEST_CORNER_REL)
print("Merged test GT:", tinyperson_data_root / TINYPERSON_TEST_MERGED_REL)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "albumentations",
        "kornia",
        "timm",
        "huggingface_hub",
        "opencv-python-headless",
        "pycocotools",
        "pillow",
        "pyyaml",
        "ultralytics-thop",
    ],
    check=True,
)

In [ ]:
# ================================================================
# GITHUB TOKEN — live-only cell.
# Dán token vào đây nếu repo GitHub private hoặc cần tránh auth prompt.
# Không push/export notebook public khi cell này chứa token thật.
# ================================================================
import os as _yolo_token_os

# TODO: paste token here in live if needed, e.g. "github_pat_...".
yolo_github_token = "[REDACTED_SECRET]"

if yolo_github_token:
    _yolo_token_os.environ["GITHUB_TOKEN"] = yolo_github_token

print("GITHUB_TOKEN configured for GitHub clone/pull:", bool(_yolo_token_os.environ.get("GITHUB_TOKEN")))

In [ ]:
# ================================================================
# LOAD THE MAIN YOLO_FTSC CODEBASE FROM GITHUB — full repo sync.
# TinyPerson is downloaded/extracted by the cells above.
# ================================================================
from pathlib import Path as _YoloCodePath
import os as _yolo_code_os
import shutil as _yolo_code_shutil
import subprocess as _yolo_code_subprocess
import time as _yolo_code_time
from urllib.parse import urlparse as _yolo_urlparse, urlunparse as _yolo_urlunparse


yolo_github_repo_url = "https://github.com/TLMHoang-AI/YOLO_FTSC.git"
yolo_code_dir = _YoloCodePath("/marimo/yolo_code")
yolo_github_branch = "main"


def _yolo_tokenized_url(_url: str) -> str:
    _token = _yolo_code_os.environ.get("GITHUB_TOKEN")
    if not _token or not _url.startswith("https://github.com/"):
        return _url
    _parsed = _yolo_urlparse(_url)
    return _yolo_urlunparse(
        _parsed._replace(netloc=f"x-access-token:{_token}@{_parsed.netloc}")
    )


def _yolo_redact(_parts):
    _token = _yolo_code_os.environ.get("GITHUB_TOKEN", "")
    if not _token:
        return _parts
    return [_part.replace(_token, "<GITHUB_TOKEN>") for _part in _parts]


def _yolo_git(_args, *, _cwd=None, _check=True):
    _env = dict(_yolo_code_os.environ)
    _env["GIT_TERMINAL_PROMPT"] = "0"
    _result = _yolo_code_subprocess.run(
        ["git", *_args],
        cwd=_cwd,
        env=_env,
        text=True,
        capture_output=True,
    )
    if _result.stdout:
        print(_result.stdout)
    if _result.stderr:
        print(_result.stderr)
    if _check and _result.returncode != 0:
        _safe = " ".join(_yolo_redact(["git", *_args]))
        raise RuntimeError(f"Git command failed: {_safe}")
    return _result


_yolo_core_required_markers = [
    yolo_code_dir / "train_tinyperson_scripts/train_all_tinyperson.py",
    yolo_code_dir / "utils/hf_checkpoint_recovery.py",
    yolo_code_dir / "utils/ftsc_completion_planner.py",
    yolo_code_dir / "utils/hf_result_archive.py",
    yolo_code_dir / "train_tinyperson_scripts/train_tinyperson_ftsc_causal_suite.py",
    yolo_code_dir / "models_related/models_config/yolov8/tinyperson/yolov8n_p2_tinyperson_h2_noftsc_nomosaic.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/tinyperson/yolov8n_p2_tinyperson_ftsc_h2_clean_edge025_nomosaic.yaml",
    yolo_code_dir / "vendor/tinyperson_cocoeval.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/engine/trainer.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/ftsc.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/nn/modules/edge_cue.py",
    yolo_code_dir / "models_related/ultralytics/ultralytics/utils/loss.py",
    yolo_code_dir / "models_related/models_config/yolov8/levir/yolov8n_p2_levir_ftsc_nomosaic_h2_p2_p3.yaml",
    yolo_code_dir / "models_related/models_config/yolov8/levir/yolov8n_p2_levir_ftsc_h2_edge_stab_es1_scale025.yaml",
]


def _yolo_missing_markers():
    return [str(_p) for _p in _yolo_core_required_markers if not _p.exists()]


_repo_url_for_clone = _yolo_tokenized_url(yolo_github_repo_url)

if (yolo_code_dir / ".git").is_dir():
    print("Existing yolo_code git repo found; forcing full GitHub sync...")
    _yolo_git(["remote", "set-url", "origin", _repo_url_for_clone], _cwd=yolo_code_dir)
    _yolo_git(["fetch", "origin", yolo_github_branch], _cwd=yolo_code_dir)
    # Intentional: the live experiment checkout must mirror the pushed commit.
    _yolo_git(["reset", "--hard", f"origin/{yolo_github_branch}"], _cwd=yolo_code_dir)
    _yolo_git(
        ["clean", "-fd", "-e", "runs/", "-e", "datasets/", "-e", "yolov8n.pt", "-e", "yolov9t.pt", "-e", "yolov10n.pt", "-e", "yolo11n.pt"],
        _cwd=yolo_code_dir,
    )
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
elif yolo_code_dir.exists():
    _backup = yolo_code_dir.with_name(
        f"{yolo_code_dir.name}_partial_backup_{int(_yolo_code_time.time())}"
    )
    print(f"Found non-git/partial yolo_code. Moving it to backup: {_backup}")
    _yolo_code_shutil.move(str(yolo_code_dir), str(_backup))
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git([
        "clone",
        "--branch",
        yolo_github_branch,
        _repo_url_for_clone,
        str(yolo_code_dir),
    ])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)
else:
    print("Cloning FULL YOLO_FTSC repo from GitHub...")
    _yolo_git([
        "clone",
        "--branch",
        yolo_github_branch,
        _repo_url_for_clone,
        str(yolo_code_dir),
    ])
    _yolo_git(["remote", "set-url", "origin", yolo_github_repo_url], _cwd=yolo_code_dir)

_missing_yolo = _yolo_missing_markers()
if _missing_yolo:
    raise FileNotFoundError(
        "YOLO FTSC sync is missing required causal-suite source files. Push them to the selected GitHub branch before Run all:\n- "
        + "\n- ".join(_missing_yolo)
    )

print("Main YOLO_FTSC codebase ready:", yolo_code_dir)
print("Latest commit:")
_yolo_git(["log", "--oneline", "-1"], _cwd=yolo_code_dir)

In [ ]:
# ================================================================
# LOAD CANONICAL TINYPERSON PREPARATION FROM THE FRESH yolo_code.
# The journal completion cell prepares the seed datasets automatically.
# This cell validates the preparation API but does not materialize all 3 seeds.
# ================================================================
from pathlib import Path as _TpPrepPath
import importlib.util as _tp_importlib_util
import sys as _tp_prepare_sys

TINYPERSON_PREPARE_SEEDS = (42, 43, 44)
tinyperson_runner_file = (
    _TpPrepPath(yolo_code_dir)
    / "train_tinyperson_scripts/train_all_tinyperson.py"
).resolve()
tinyperson_prepared_root = _TpPrepPath(
    _TpPrepPath(yolo_code_dir) / "datasets"
).resolve()

if not tinyperson_runner_file.is_file():
    raise FileNotFoundError(tinyperson_runner_file)

_yolo_root_for_tiny = _TpPrepPath(yolo_code_dir).resolve()
if str(_yolo_root_for_tiny) not in _tp_prepare_sys.path:
    _tp_prepare_sys.path.insert(0, str(_yolo_root_for_tiny))

_tp_module_name = "_live_main_yolo_tinyperson_prepare"
_tp_prepare_sys.modules.pop(_tp_module_name, None)
_tp_spec = _tp_importlib_util.spec_from_file_location(
    _tp_module_name,
    tinyperson_runner_file,
)
if _tp_spec is None or _tp_spec.loader is None:
    raise ImportError(f"Cannot import TinyPerson runner: {tinyperson_runner_file}")
tinyperson_prepare_module = _tp_importlib_util.module_from_spec(_tp_spec)
_tp_prepare_sys.modules[_tp_module_name] = tinyperson_prepare_module
_tp_spec.loader.exec_module(tinyperson_prepare_module)

for _required_function in (
    "prepare_test_set",
    "prepare_seed_dataset",
    "load_corner_annotations",
    "write_corner_crop",
    "evaluate_tiny_benchmark",
    "evaluate",
):
    if not callable(getattr(tinyperson_prepare_module, _required_function, None)):
        raise RuntimeError(
            f"Pulled TinyPerson pipeline is missing {_required_function}()"
        )

tinyperson_prepared_root.mkdir(parents=True, exist_ok=True)
print("TinyPerson canonical preparation API ready:")
print("  source root:", tinyperson_data_root)
print("  prepared root:", tinyperson_prepared_root)
print("  canonical seeds:", TINYPERSON_PREPARE_SEEDS)
print("  seed materialization is automatic in the journal completion cell")


In [ ]:
# ================================================================
# LOAD THE DEDICATED TINYPERSON CAUSAL-ABLATION RUNNER FROM yolo_code.
# This cell imports only experiment orchestration; it does not train.
# ================================================================
from pathlib import Path as _TpCausalPath
import importlib.util as _tp_causal_importlib_util
import sys as _tp_causal_sys

TINYPERSON_CAUSAL_VARIANTS = (
    "H2_NOFTSC_NM",
    "CLEAN_EDGE_NM",
    "H2_MOSAIC",
    "ES1_MOSAIC",
)
tinyperson_causal_runner_file = (
    _TpCausalPath(yolo_code_dir)
    / "train_tinyperson_scripts/train_tinyperson_ftsc_causal_suite.py"
).resolve()

if not tinyperson_causal_runner_file.is_file():
    raise FileNotFoundError(
        "Missing the dedicated TinyPerson causal-suite runner in the synced "
        f"yolo_code checkout: {tinyperson_causal_runner_file}. Push/sync the "
        "new runner and model YAMLs before running this notebook."
    )

_yolo_root_for_causal = _TpCausalPath(yolo_code_dir).resolve()
if str(_yolo_root_for_causal) not in _tp_causal_sys.path:
    _tp_causal_sys.path.insert(0, str(_yolo_root_for_causal))

# Load from the exact synced file so an older cached module cannot be reused.
_tp_causal_module_name = "_live_main_yolo_tinyperson_ftsc_causal_suite"
_tp_causal_sys.modules.pop(_tp_causal_module_name, None)
_tp_causal_spec = _tp_causal_importlib_util.spec_from_file_location(
    _tp_causal_module_name,
    tinyperson_causal_runner_file,
)
if _tp_causal_spec is None or _tp_causal_spec.loader is None:
    raise ImportError(f"Cannot import causal-suite runner: {tinyperson_causal_runner_file}")
tinyperson_causal_suite_module = _tp_causal_importlib_util.module_from_spec(
    _tp_causal_spec
)
_tp_causal_sys.modules[_tp_causal_module_name] = tinyperson_causal_suite_module
_tp_causal_spec.loader.exec_module(tinyperson_causal_suite_module)

_actual_variants = tuple(tinyperson_causal_suite_module.VARIANTS)
if _actual_variants != TINYPERSON_CAUSAL_VARIANTS:
    raise RuntimeError(
        "Causal-suite registry drifted: "
        f"{_actual_variants} != {TINYPERSON_CAUSAL_VARIANTS}"
    )
if {"H2_NM", "ES1_NM"} & set(_actual_variants):
    raise RuntimeError("Historical H2_NM/ES1_NM must never be active training variants")
if _TpCausalPath(tinyperson_causal_suite_module.ROOT).resolve() != _yolo_root_for_causal:
    raise RuntimeError("Causal runner did not resolve to the synced main yolo_code checkout")

for _required_api in (
    "parse_args",
    "validate_args",
    "source_preflight",
    "model_preflight",
    "lock_seed_dataset",
    "run_one",
    "write_summaries",
):
    if not callable(getattr(tinyperson_causal_suite_module, _required_api, None)):
        raise RuntimeError(f"Causal runner is missing {_required_api}()")

print("TinyPerson causal-suite runner loaded from:", tinyperson_causal_runner_file)
print("Active new variants:", ", ".join(TINYPERSON_CAUSAL_VARIANTS))
print("Historical H2_NM/ES1_NM: read-only aggregation controls")

## TinyPerson FTSC journal metric completion

Run All automatically performs HF checkpoint discovery, strict historical provenance checks, best.pt restore, TinyBenchmark merged-original mAP50:95 evaluation, six-core validation, 3-seed aggregation and non-overwriting HF publication. No training is authorized. The journal default is H2_NOFTSC_NM, H2_NM (historical H2) and H2_MOSAIC, seeds 42/43/44. LMSCA, OACP, CP2, R2, R4 and Edge cases are outside the automatic matrix.

The separate legacy causal and alpha training gates remain disabled. Status is persisted after each seed at runs/tinyperson_ftsc_journal_metric_completion_v1/completion_status.json. If an archive is missing or provenance fails, it stays BLOCKED; incomplete results are not aggregated or published.


In [ ]:
RUN_TINYPERSON_FTSC_CAUSAL_SUITE = False


In [ ]:
# Dedicated TinyPerson alpha launcher. Disabled and preflight-only by default.
RUN_TINYPERSON_FTSC_ALPHA = False
TINYPERSON_FTSC_ALPHA_PHASE = "preflight"  # preflight/train/evaluate/validate/aggregate/archive/backfill-historical
TINYPERSON_FTSC_ALPHA_CASES = ("ALPHA4_rerun", "ALPHA8_rerun")
TINYPERSON_FTSC_ALPHA_SEEDS = (42, 43, 44)
TINYPERSON_FTSC_ALPHA_WORKERS = 8
TINYPERSON_FTSC_ALPHA_DEVICE = "cuda"
TINYPERSON_FTSC_ALPHA_LIVE_MODEL_PREFLIGHT = True
ARCHIVE_TINYPERSON_FTSC_ALPHA = False
UPLOAD_TINYPERSON_FTSC_ALPHA_AFTER_EACH_RUN = True
if RUN_TINYPERSON_FTSC_ALPHA:
    from pathlib import Path as _TpAlphaPath
    import importlib.util as _tp_alpha_importlib_util
    import sys as _tp_alpha_sys
    _tp_alpha_file = _TpAlphaPath(yolo_code_dir) / "train_tinyperson_scripts/train_tinyperson_ftsc_alpha_sensitivity_v1.py"
    if not _tp_alpha_file.is_file():
        raise FileNotFoundError(f"Missing synced alpha runner: {_tp_alpha_file}. Push the local implementation, then rerun yolo_code_sync.")
    _tp_alpha_module_name = "_live_tinyperson_ftsc_alpha_sensitivity_v1"
    _tp_alpha_sys.modules.pop(_tp_alpha_module_name, None)
    _tp_alpha_spec = _tp_alpha_importlib_util.spec_from_file_location(_tp_alpha_module_name, _tp_alpha_file)
    if _tp_alpha_spec is None or _tp_alpha_spec.loader is None:
        raise ImportError(f"Cannot load alpha runner: {_tp_alpha_file}")
    _tp_alpha = _tp_alpha_importlib_util.module_from_spec(_tp_alpha_spec)
    _tp_alpha_sys.modules[_tp_alpha_module_name] = _tp_alpha
    _tp_alpha_spec.loader.exec_module(_tp_alpha)
    _tp_alpha_cli = [
        "--phase", TINYPERSON_FTSC_ALPHA_PHASE,
        "--data-root", str(tinyperson_data_root),
        "--dataset-root", str(_TpAlphaPath(yolo_code_dir) / "datasets"),
        "--project", str(_TpAlphaPath(yolo_code_dir) / "runs/tinyperson_ftsc_position_alpha_h2_v1"),
        "--pretrained", str(_TpAlphaPath(yolo_code_dir) / "yolov8n.pt"),
        "--workers", str(TINYPERSON_FTSC_ALPHA_WORKERS),
        "--device", TINYPERSON_FTSC_ALPHA_DEVICE,
        "--cases", *TINYPERSON_FTSC_ALPHA_CASES,
        "--seeds", *[str(_seed) for _seed in TINYPERSON_FTSC_ALPHA_SEEDS],
    ]
    if TINYPERSON_FTSC_ALPHA_LIVE_MODEL_PREFLIGHT:
        _tp_alpha_cli.append("--live-model-preflight")
    if TINYPERSON_FTSC_ALPHA_PHASE != "preflight":
        from utils.hf_result_archive import HFResultArchive as _TpAlphaArchive
        _tp_alpha_archive = _TpAlphaArchive(repo_id="TLMHoang/FTSC_results", token=HF_TOKEN, cache_dir="/marimo/hf_results_cache/tinyperson_ftsc_position_alpha_h2_v1")
        _tp_alpha_archive.restore_completed_runs(dataset_slug=_tp_alpha.NAMESPACE, project=_TpAlphaPath(yolo_code_dir) / "runs" / _tp_alpha.NAMESPACE, variants=TINYPERSON_FTSC_ALPHA_CASES, seeds=TINYPERSON_FTSC_ALPHA_SEEDS, run_name_fn=lambda _case, _seed: f"seed_{_seed}_corner_sw640_sh512", required_artifacts=(*_tp_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _tp_alpha.METRICS_ARTIFACT, _tp_alpha.COMPLETE_ARTIFACT))
    if TINYPERSON_FTSC_ALPHA_PHASE == "train" and UPLOAD_TINYPERSON_FTSC_ALPHA_AFTER_EACH_RUN:
        _tp_alpha_case_index = _tp_alpha_cli.index("--cases")
        _tp_alpha_base_cli = _tp_alpha_cli[:_tp_alpha_case_index]
        for _case in TINYPERSON_FTSC_ALPHA_CASES:
            for _seed in TINYPERSON_FTSC_ALPHA_SEEDS:
                _run = _TpAlphaPath(yolo_code_dir) / "runs" / _tp_alpha.NAMESPACE / _case / f"seed_{_seed}_corner_sw640_sh512"
                _training_ready = all((_run / _relative).is_file() for _relative in _tp_alpha.TRAIN_ARTIFACTS)
                if not _training_ready:
                    _tp_alpha.main([*_tp_alpha_base_cli, "--cases", _case, "--seeds", str(_seed)])
                _evaluate_cli = [*_tp_alpha_base_cli, "--cases", _case, "--seeds", str(_seed)]
                _evaluate_cli[1] = "evaluate"
                if not (_run / _tp_alpha.COMPLETE_ARTIFACT).is_file():
                    _tp_alpha.main(_evaluate_cli)
                _validate_cli = [*_evaluate_cli]
                _validate_cli[1] = "validate"
                _tp_alpha.main(_validate_cli)
                _tp_alpha_archive.publish_run(dataset_slug=_tp_alpha.NAMESPACE, variant=_case, seed=_seed, run_name=_run.name, run_dir=_run, required_artifacts=(*_tp_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _tp_alpha.METRICS_ARTIFACT, _tp_alpha.COMPLETE_ARTIFACT))
                print(f"TRAIN+EVAL+VALIDATE+UPLOAD complete: {_case}/seed_{_seed}", flush=True)
    else:
        _tp_alpha.main(_tp_alpha_cli)
    if ARCHIVE_TINYPERSON_FTSC_ALPHA and TINYPERSON_FTSC_ALPHA_PHASE == "archive":
        for _case in TINYPERSON_FTSC_ALPHA_CASES:
            for _seed in TINYPERSON_FTSC_ALPHA_SEEDS:
                _run = _TpAlphaPath(yolo_code_dir) / "runs" / _tp_alpha.NAMESPACE / _case / f"seed_{_seed}_corner_sw640_sh512"
                _tp_alpha_archive.publish_run(dataset_slug=_tp_alpha.NAMESPACE, variant=_case, seed=_seed, run_name=_run.name, run_dir=_run, required_artifacts=(*_tp_alpha.TRAIN_ARTIFACTS, "experiment_manifest.json", _tp_alpha.METRICS_ARTIFACT, _tp_alpha.COMPLETE_ARTIFACT))


In [ ]:
# Journal-only TinyPerson metric completion. ONE run: HF inventory -> verified
# checkpoint recovery -> TinyBenchmark original-image evaluation -> aggregate -> publish.
# No training, no phase edits, no LMSCA/OACP/CP2/R2/R4.
RUN_TINYPERSON_FTSC_JOURNAL_COMPLETION = False
TINYPERSON_JOURNAL_CASES = ("H2_NOFTSC_NM", "H2_NM", "H2_MOSAIC")
TINYPERSON_JOURNAL_SEEDS = (42, 43, 44)

if RUN_TINYPERSON_FTSC_JOURNAL_COMPLETION:
    import argparse as _tj_argparse
    import csv as _tj_csv
    import hashlib as _tj_hashlib
    import json as _tj_json
    import math as _tj_math
    import os as _tj_os
    import shutil as _tj_shutil
    import statistics as _tj_statistics
    from pathlib import Path as _TJPath
    import yaml as _tj_yaml
    from huggingface_hub import hf_hub_download as _tj_hf_download
    from utils.hf_checkpoint_recovery import HFCheckpointRecovery as _TJRecovery
    from utils.ftsc_completion_planner import validate_six_core as _tj_validate_core
    from utils.hf_result_archive import HFResultArchive as _TJArchive

    _tj_repo = "TLMHoang/FTSC_results"
    _tj_namespace = "tinyperson_ftsc_journal_metric_completion_v1"
    _tj_project = _TJPath(yolo_code_dir) / "runs" / _tj_namespace
    _tj_cache = _TJPath("/marimo/hf_results_cache/tinyperson_journal_completion")
    _tj_project.mkdir(parents=True, exist_ok=True)
    _tj_cache.mkdir(parents=True, exist_ok=True)
    _tj_cases = tuple(TINYPERSON_JOURNAL_CASES)
    _tj_seeds = tuple(TINYPERSON_JOURNAL_SEEDS)
    _tj_allowed_cases = ("H2_NOFTSC_NM", "H2_NM", "H2_MOSAIC")
    if _tj_cases != _tj_allowed_cases or _tj_seeds != (42, 43, 44):
        raise RuntimeError("Journal recovery is frozen to 3 canonical FTSC cases x 3 seeds")
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN required for HF-first journal completion")
    _tj_os.environ["HF_TOKEN"] = HF_TOKEN
    _tj_sources = {
        "H2_NOFTSC_NM": {
            "yaml": tinyperson_causal_suite_module.H2_NOFTSC_CONFIG,
            "aliases": ("H2_NOFTSC_NM",),
            "prefixes": ("tinyperson_ftsc_causal_suite", "tinyperson"),
            "mosaic": 0.0, "close_mosaic": 0, "ftsc": False,
        },
        "H2_NM": {
            "yaml": tinyperson_causal_suite_module.H2_CONFIG,
            "aliases": ("H2", "H2_NM"),
            "prefixes": ("tinyperson_ftsc_h2_es1", "tinyperson"),
            "mosaic": 0.0, "close_mosaic": 0, "ftsc": True,
        },
        "H2_MOSAIC": {
            "yaml": tinyperson_causal_suite_module.H2_CONFIG,
            "aliases": ("H2_MOSAIC",),
            "prefixes": ("tinyperson_ftsc_causal_suite", "tinyperson"),
            "mosaic": 1.0, "close_mosaic": 10, "ftsc": True,
        },
    }
    _tj_metric_keys = {
        "AP50": "test_merged/AP50",
        "mAP50-95": "test_merged/mAP50-95",
        "AP50-Small": "test_merged/AP50-Small",
        "parameters": "model/parameters",
        "GFLOPs": "model/GFLOPs",
        "inference_ms": "test_speed/inference_ms_per_image",
    }
    _tj_seed_run = lambda case, seed: _tj_project / case / f"seed_{seed}"
    _tj_progress_path = _tj_project / "completion_status.json"

    def _tj_sha(path):
        digest = _tj_hashlib.sha256()
        with _TJPath(path).open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def _tj_read(path):
        value = _tj_json.loads(_TJPath(path).read_text(encoding="utf-8"))
        if not isinstance(value, dict):
            raise ValueError(f"Expected JSON object at {path}")
        return value

    def _tj_write(path, value):
        path = _TJPath(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".write_tmp")
        if temporary.exists():
            raise RuntimeError(f"Stale temporary file: {temporary}")
        temporary.write_text(
            _tj_json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n",
            encoding="utf-8",
        )
        temporary.replace(path)

    def _tj_provenance_paths():
        root = _TJPath(yolo_code_dir)
        names = {
            "tiny_workflow": "train_tinyperson_scripts/train_all_tinyperson.py",
            "tiny_evaluator": "vendor/tinyperson_cocoeval.py",
            "tiny_causal": "train_tinyperson_scripts/train_tinyperson_ftsc_causal_suite.py",
            "checkpoint_recovery": "utils/hf_checkpoint_recovery.py",
            "ftsc": "models_related/ultralytics/ultralytics/nn/modules/ftsc.py",
            "loss": "models_related/ultralytics/ultralytics/utils/loss.py",
        }
        return {name: _tj_sha(root / relative) for name, relative in names.items()}

    _tj_evaluator_sources = _tj_provenance_paths()
    _tj_source_check = _TJPath(tinyperson_runner_file).read_text(encoding="utf-8")
    for _tj_required in (
        "test_merged/mAP50-95",
        "np.linspace(0.50, 0.95, 10)",
        "nms_detections(items, iou_threshold=0.5)",
        "conf=0.001", "iou=0.5", "max_det=300",
        "test_speed/inference_ms_per_image",
    ):
        if _tj_required not in _tj_source_check:
            raise RuntimeError(f"Synced TinyPerson merged evaluator is outdated: {_tj_required}")

    def _tj_verify_metadata(case, seed, metadata):
        spec = _tj_sources[case]
        if "config.yaml" not in metadata or "args.yaml" not in metadata:
            return False, "Archived config.yaml/args.yaml missing"
        try:
            model_cfg = _tj_yaml.safe_load(_TJPath(metadata["config.yaml"]).read_text())
            canonical_cfg = _tj_yaml.safe_load(_TJPath(spec["yaml"]).read_text())
            if model_cfg != canonical_cfg:
                return False, "Model graph/FTSC YAML differs from frozen journal case"
            saved = _tj_yaml.safe_load(_TJPath(metadata["args.yaml"]).read_text())
            if not isinstance(saved, dict):
                return False, "Invalid historical training arguments"
            frozen = {
                "seed": seed, "imgsz": 640, "batch": 8, "epochs": 100,
                "patience": 20, "deterministic": True, "amp": True,
                "mosaic": spec["mosaic"], "close_mosaic": spec["close_mosaic"],
                "optimizer": "AdamW", "lr0": 0.002, "fitness_metric": "map50_95",
            }
            for name, expected in frozen.items():
                actual = saved.get(name)
                if isinstance(expected, bool):
                    valid = type(actual) is bool and actual is expected
                elif isinstance(expected, (int, float)):
                    valid = (isinstance(actual, (int, float))
                             and not isinstance(actual, bool) and actual == expected)
                else:
                    valid = actual == expected
                if not valid:
                    return False, f"Historical recipe mismatch: {name}: {actual!r} != {expected!r}"
            for name, expected in (("mixup", 0), ("cutmix", 0), ("copy_paste", 0)):
                if saved.get(name, expected) != expected:
                    return False, f"Unexpected augmentation: {name}"
            dataset_path = str(saved.get("data", ""))
            expected_folder = f"tinyperson_seed_{seed}_corner_sw640_sh512/"
            if expected_folder not in dataset_path or not dataset_path.endswith("tinyperson.yaml"):
                return False, "Historical seed-specific corner dataset lineage unclear"
            for file_name in ("experiment_manifest.json", "run_summary.json"):
                if file_name not in metadata:
                    continue
                record = _tj_read(metadata[file_name])
                historical_case = record.get("variant", record.get("case"))
                if historical_case is not None and historical_case not in spec["aliases"]:
                    return False, f"{file_name}: wrong historical case"
                if record.get("seed") is not None and str(record["seed"]) != str(seed):
                    return False, f"{file_name}: wrong historical seed"
                training = record.get("training")
                if isinstance(training, dict):
                    for key in ("mosaic", "close_mosaic", "seed"):
                        if key in training and training[key] != frozen[key]:
                            return False, f"{file_name}: wrong training {key}"
            return True, (
                "YAML/seed/corner path/frozen training args agree. "
                "Original split-content SHA and pretrained-source hash not established by args alone."
            )
        except (OSError, ValueError, TypeError, KeyError) as exc:
            return False, f"Unverifiable archived source: {type(exc).__name__}: {exc}"

    def _tj_valid_completion(case, seed, checkpoint_sha=None):
        run = _tj_seed_run(case, seed)
        metric_path = run / "evaluation_metrics_completion_v1.json"
        source_path = run / "completion_source.json"
        recovered = run / "recovery_source.json"
        checkpoint = run / "weights/best.pt"
        if not all(p.is_file() for p in (metric_path, source_path, recovered, checkpoint)):
            return None
        try:
            completion = _tj_read(source_path)
            source = _tj_read(recovered)
            metrics = _tj_read(metric_path)
            actual_sha = _tj_sha(checkpoint)
            current_seed_yaml = _TJPath(tinyperson_prepared_root) / (
                f"tinyperson_seed_{seed}_corner_sw640_sh512/tinyperson.yaml"
            )
            current_manifest = current_seed_yaml.parent / "corner_manifest.json"
            if (not current_seed_yaml.is_file() or not current_manifest.is_file()
                    or completion.get("dataset_yaml_sha256") != _tj_sha(current_seed_yaml)
                    or completion.get("prepared_seed_manifest_sha256") != _tj_sha(current_manifest)):
                return None
            if (completion.get("case") != case or completion.get("seed") != seed
                    or completion.get("checkpoint_sha256") != actual_sha
                    or source.get("checkpoint_sha256") != actual_sha
                    or (checkpoint_sha and checkpoint_sha != actual_sha)
                    or completion.get("metrics_sha256") != _tj_sha(metric_path)
                    or completion.get("evaluator_sources") != _tj_evaluator_sources
                    or completion.get("model_yaml_sha256") != _tj_sha(_tj_sources[case]["yaml"])
                    or completion.get("evaluator_contract") != "tinybenchmark_merged_original_iou50_95_v2"):
                return None
            if _tj_validate_core(metrics, _tj_metric_keys):
                return None
            if metrics.get("test_merged/available") != 1.0:
                return None
            return metrics
        except (ValueError, OSError, TypeError):
            return None

    _tj_status = {
        "workflow": _tj_namespace, "repo_id": _tj_repo,
        "cases": list(_tj_cases), "seeds": list(_tj_seeds),
        "training": "NEVER", "results": [], "complete": False,
    }

    def _tj_record(case, seed, status, **details):
        row = {"case": case, "seed": seed, "status": status, **details}
        _tj_status["results"].append(row)
        _tj_status["processed"] = len(_tj_status["results"])
        _tj_write(_tj_progress_path, _tj_status)
        print(f"[TINY JOURNAL {_tj_status['processed']}/9] "
              f"{case}/seed_{seed} => {status}", flush=True)
        return row

    _tj_agent = _TJRecovery(_tj_repo, token=HF_TOKEN)
    _tj_prefixes = sorted({p for spec in _tj_sources.values() for p in spec["prefixes"]})
    print("[TINY JOURNAL] HF source inventory (pinned revision); 9 historical runs", flush=True)
    try:
        _tj_inventory = _tj_agent.inventory(_tj_prefixes)
    except Exception as _tj_exc:
        _tj_status["fatal"] = "BLOCKED_HF_UNAVAILABLE: " + str(_tj_exc).replace(HF_TOKEN, "<HF_TOKEN>")
        _tj_write(_tj_progress_path, _tj_status)
        raise RuntimeError(_tj_status["fatal"]) from None
    _tj_status["hf_source_revision"] = _tj_inventory["revision"]
    _tj_write(_tj_progress_path, _tj_status)

    _tj_picks = {}
    for _tj_case in _tj_cases:
        _tj_spec = _tj_sources[_tj_case]
        for _tj_seed in _tj_seeds:
            _tj_named = _tj_agent.candidates(
                _tj_inventory, case=_tj_case, seed=_tj_seed,
                aliases=_tj_spec["aliases"], prefixes=_tj_spec["prefixes"],
            )
            _tj_candidates = _tj_named or _tj_agent.candidates(
                _tj_inventory, case=_tj_case, seed=_tj_seed,
                aliases=_tj_spec["aliases"], prefixes=_tj_spec["prefixes"], match_case=False,
            )
            _tj_accepted = []
            _tj_rejections = []
            for _tj_candidate in _tj_candidates:
                try:
                    _tj_metadata = _tj_agent.download_metadata(_tj_candidate, _tj_cache)
                    _tj_ok, _tj_reason = _tj_verify_metadata(_tj_case, _tj_seed, _tj_metadata)
                    if _tj_ok:
                        _tj_accepted.append((
                            _tj_candidate, _tj_agent.marker_checksum(_tj_metadata), _tj_reason,
                            _tj_metadata.get("experiment_manifest.json"),
                        ))
                    else:
                        _tj_rejections.append({
                            "hf_path": _tj_candidate.checkpoint_path, "reason": _tj_reason,
                        })
                except Exception as _tj_exc:
                    _tj_rejections.append({
                        "hf_path": _tj_candidate.checkpoint_path,
                        "reason": f"{type(_tj_exc).__name__}: {str(_tj_exc).replace(HF_TOKEN, '<HF_TOKEN>')}",
                    })
            _tj_unique_paths = {candidate.checkpoint_path for candidate, *_ in _tj_accepted}
            if len(_tj_unique_paths) != 1:
                _tj_picks[(_tj_case, _tj_seed)] = None
                _tj_record(
                    _tj_case, _tj_seed,
                    "BLOCKED_NO_VERIFIED_SOURCE" if not _tj_accepted else "BLOCKED_AMBIGUOUS_CHECKPOINT",
                    candidates=len(_tj_candidates), rejected=_tj_rejections[:12],
                )
            else:
                _tj_picks[(_tj_case, _tj_seed)] = _tj_accepted[0]
                print(f"[TINY JOURNAL] VERIFIED {_tj_case}/seed_{_tj_seed}: "
                      f"{_tj_accepted[0][0].checkpoint_path}", flush=True)

    # Reuse the native original-image corner/window pipeline, not Varroa evaluation.
    _tj_ready = [key for key, candidate in _tj_picks.items() if candidate is not None]
    if _tj_ready:
        _tj_eval_args = _tj_argparse.Namespace(imgsz=640, batch_size=8, device="cuda", workers=8)
        _tj_test_out = tinyperson_prepare_module.prepare_test_set(
            _TJPath(tinyperson_data_root), _TJPath(tinyperson_prepared_root)
        )
        _tj_seed_data = {}
        for _tj_seed in _tj_seeds:
            if any(seed == _tj_seed for _, seed in _tj_ready):
                _tj_seed_dir = tinyperson_prepare_module.prepare_seed_dataset(
                    _TJPath(tinyperson_data_root), _TJPath(tinyperson_prepared_root),
                    _tj_test_out, _tj_seed,
                )
                _tj_seed_yaml = _tj_seed_dir / "tinyperson.yaml"
                _tj_seed_manifest = _tj_seed_dir / "corner_manifest.json"
                if not _tj_seed_yaml.is_file() or not _tj_seed_manifest.is_file():
                    raise RuntimeError(f"Prepared TinyPerson dataset incomplete: seed {_tj_seed}")
                _tj_seed_data[_tj_seed] = (_tj_seed_yaml, _tj_seed_manifest)

    for _tj_case in _tj_cases:
        for _tj_seed in _tj_seeds:
            _tj_chosen = _tj_picks[(_tj_case, _tj_seed)]
            if _tj_chosen is None:
                continue
            _tj_candidate, _tj_marker_sha, _tj_evidence, _tj_manifest_path = _tj_chosen
            _tj_run = _tj_seed_run(_tj_case, _tj_seed)
            _tj_best = _tj_run / "weights/best.pt"
            _tj_recovery_path = _tj_run / "recovery_source.json"
            try:
                _tj_yaml_path, _tj_manifest_file = _tj_seed_data[_tj_seed]
                _tj_original_split_verified = False
                if _tj_manifest_path:
                    _tj_archive_manifest = _tj_read(_tj_manifest_path)
                    _tj_archived_yaml_sha = _tj_archive_manifest.get("data_yaml_sha256")
                    if _tj_archived_yaml_sha and _tj_archived_yaml_sha != _tj_sha(_tj_yaml_path):
                        raise RuntimeError("BLOCKED_ORIGINAL_DATASET_YAML_SHA_MISMATCH")
                    _tj_expected_train = _tj_archive_manifest.get("train_source_sha256")
                    _tj_expected_val = _tj_archive_manifest.get("val_source_sha256")
                    if bool(_tj_expected_train) != bool(_tj_expected_val):
                        raise RuntimeError("BLOCKED_PARTIAL_ARCHIVED_SPLIT_FINGERPRINT")
                    if _tj_expected_train and _tj_expected_val:
                        _tj_split = _tj_read(_tj_manifest_file)
                        _tj_actual_source_sha = {}
                        for _tj_split_name in ("train", "val"):
                            _tj_source_names = sorted({
                                str(item["file_name"]) for item in _tj_split[_tj_split_name]
                            })
                            _tj_actual_source_sha[_tj_split_name] = _tj_hashlib.sha256(
                                "\n".join(_tj_source_names).encode()
                            ).hexdigest()
                        if (_tj_actual_source_sha["train"] != _tj_expected_train
                                or _tj_actual_source_sha["val"] != _tj_expected_val):
                            raise RuntimeError("BLOCKED_ORIGINAL_IMAGE_SPLIT_FINGERPRINT_MISMATCH")
                        _tj_original_split_verified = True
                if _tj_recovery_path.is_file():
                    _tj_existing_source = _tj_read(_tj_recovery_path)
                    if _tj_existing_source.get("hf_checkpoint") != _tj_candidate.checkpoint_path:
                        raise RuntimeError("BLOCKED_SOURCE_COLLISION: previous source differs")
                    if _tj_marker_sha and _tj_existing_source.get("checkpoint_sha256") != _tj_marker_sha:
                        raise RuntimeError("BLOCKED_CHECKPOINT_SHA_CONFLICT")
                print(f"[TINY JOURNAL] RESTORE {_tj_case}/seed_{_tj_seed}", flush=True)
                _tj_restore = _tj_agent.restore(
                    _tj_candidate, destination=_tj_best, verified=True,
                    cache_dir=_tj_cache, expected_sha256=_tj_marker_sha,
                )
                if _tj_recovery_path.is_file():
                    if _tj_read(_tj_recovery_path).get("checkpoint_sha256") != _tj_restore["checkpoint_sha256"]:
                        raise RuntimeError("BLOCKED_PREVIOUS_CHECKPOINT_CONFLICT")
                _tj_restore.update(
                    case=_tj_case, seed=_tj_seed, source_verification=_tj_evidence,
                    model_yaml_sha256=_tj_sha(_tj_sources[_tj_case]["yaml"]),
                    archive_marker_sha256=_tj_marker_sha,
                )
                _tj_write(_tj_recovery_path, _tj_restore)
                _tj_cached = _tj_valid_completion(
                    _tj_case, _tj_seed, _tj_restore["checkpoint_sha256"]
                )
                if _tj_cached is not None:
                    _tj_record(_tj_case, _tj_seed, "SKIP_COMPLETE",
                               checkpoint_sha256=_tj_restore["checkpoint_sha256"])
                    continue
                print(f"[TINY JOURNAL] EVALUATE merged originals: "
                      f"{_tj_case}/seed_{_tj_seed}", flush=True)
                # A stale native evaluator cache must not masquerade as this version.
                _tj_native_cache = _tj_run / "evaluation_metrics.json"
                if _tj_native_cache.is_file():
                    # No checkpoint/protocol attestation for this cache: quarantine
                    # even when its field names happen to match the new evaluator.
                    _tj_stale = _tj_run / (
                        "evaluation_metrics.stale_" + _tj_sha(_tj_native_cache)[:12] + ".json"
                    )
                    if _tj_stale.exists():
                        raise RuntimeError("BLOCKED_STALE_EVALUATOR_CACHE_COLLISION")
                    _tj_native_cache.replace(_tj_stale)
                _tj_metrics = tinyperson_prepare_module.evaluate(
                    _tj_run, _tj_yaml_path, _tj_test_out,
                    _TJPath(tinyperson_data_root), _tj_eval_args,
                )
                if (_tj_metrics.get("test_merged/evaluator_contract") !=
                        "tinybenchmark_merged_original_iou50_95_v2"
                        or _tj_metrics.get("test_merged/available") != 1.0):
                    raise RuntimeError("BLOCKED_MERGED_ORIGINAL_EVALUATOR_UNAVAILABLE")
                tinyperson_causal_suite_module.local_ultralytics()
                from ultralytics import YOLO as _TJYOLO
                from ultralytics.utils.torch_utils import get_flops as _tj_get_flops
                _tj_model = _TJYOLO(_tj_best)
                _tj_det = _tj_model.model.model[-1]
                _tj_ftsc = getattr(_tj_det, "ftsc_calibrator", None)
                if ((_tj_ftsc is not None) is not _tj_sources[_tj_case]["ftsc"]
                        or list(_tj_det.f) != [19, 22]
                        or [float(s) for s in _tj_det.stride.detach().cpu().tolist()] != [4.0, 8.0]):
                    raise RuntimeError("BLOCKED_CHECKPOINT_ARCHITECTURE_MISMATCH")
                _tj_metrics = dict(_tj_metrics)
                _tj_metrics["model/parameters"] = int(
                    sum(p.numel() for p in _tj_model.model.parameters())
                )
                _tj_metrics["model/GFLOPs"] = float(_tj_get_flops(_tj_model.model, imgsz=640))
                _tj_missing = _tj_validate_core(_tj_metrics, _tj_metric_keys)
                if _tj_missing:
                    raise RuntimeError(f"METRIC_INCOMPLETE: {list(_tj_missing)}")
                import torch as _tj_torch
                _tj_gpu = (_tj_torch.cuda.get_device_name(0)
                           if _tj_torch.cuda.is_available() else "cpu")
                _tj_metrics.update({
                    "benchmark/gpu_name": _tj_gpu,
                    "benchmark/imgsz": 640,
                    "benchmark/corner_geometry": "sw640_sh512",
                    "benchmark/batch": 8,
                    "benchmark/device": "cuda",
                    "benchmark/precision": "ultralytics_val_default",
                    "benchmark/backend": "pytorch",
                    "benchmark/torch_version": str(_tj_torch.__version__),
                    "benchmark/cuda_runtime": str(_tj_torch.version.cuda),
                    "benchmark/gflops_method": "vendored_ultralytics_get_flops_thop",
                    "benchmark/speed_unit": "ms_per_window_inference_only",
                })
                _tj_completion_metrics = _tj_run / "evaluation_metrics_completion_v1.json"
                _tj_write(_tj_completion_metrics, _tj_metrics)
                _tj_write(_tj_run / "completion_source.json", {
                    "case": _tj_case, "seed": _tj_seed, "status": "COMPLETE",
                    "checkpoint_sha256": _tj_restore["checkpoint_sha256"],
                    "metrics_sha256": _tj_sha(_tj_completion_metrics),
                    "evaluator_sources": _tj_evaluator_sources,
                    "evaluator_contract": "tinybenchmark_merged_original_iou50_95_v2",
                    "model_yaml_sha256": _tj_sha(_tj_sources[_tj_case]["yaml"]),
                    "prepared_seed_manifest_sha256": _tj_sha(_tj_manifest_file),
                    "dataset_yaml_sha256": _tj_sha(_tj_yaml_path),
                    "original_split_sha_verified": _tj_original_split_verified,
                    "performance": "inference_only_ms_per_window",
                    "training": "never_retrained",
                })
                _tj_record(_tj_case, _tj_seed, "COMPLETE",
                           checkpoint_sha256=_tj_restore["checkpoint_sha256"])
                del _tj_model
            except Exception as _tj_exc:
                _tj_record(
                    _tj_case, _tj_seed, "BLOCKED_EVAL_OR_RECOVERY",
                    reason=f"{type(_tj_exc).__name__}: {str(_tj_exc).replace(HF_TOKEN, '<HF_TOKEN>')}",
                )

    # Journal gate: never aggregate a partial or mixed-generation seed set.
    _tj_rows = []
    for _tj_case in _tj_cases:
        for _tj_seed in _tj_seeds:
            _tj_run = _tj_seed_run(_tj_case, _tj_seed)
            _tj_verified = _tj_valid_completion(_tj_case, _tj_seed)
            if _tj_verified is not None:
                _tj_rows.append({"case": _tj_case, "seed": _tj_seed, **_tj_verified})
    _tj_status["complete"] = (
        len(_tj_rows) == 9
        and all(chosen is not None for chosen in _tj_picks.values())
        and all(sum(row["case"] == case for row in _tj_rows) == 3 for case in _tj_cases)
    )
    if not _tj_status["complete"]:
        _tj_status["final"] = "BLOCKED_PARTIAL: no journal aggregate/upload"
        _tj_status["valid_seed_runs"] = len(_tj_rows)
        _tj_write(_tj_progress_path, _tj_status)
        print(f"[TINY JOURNAL] BLOCKED_PARTIAL: {len(_tj_rows)}/9 complete. "
              f"See {_tj_progress_path}. Training has NOT been started.", flush=True)
    else:
        _tj_fields = sorted(set().union(*(set(row) for row in _tj_rows)))
        with (_tj_project / "summary_runs.csv").open("w", newline="", encoding="utf-8") as _tj_stream:
            _tj_writer = _tj_csv.DictWriter(_tj_stream, fieldnames=_tj_fields)
            _tj_writer.writeheader()
            _tj_writer.writerows(_tj_rows)
        _tj_aggregate = []
        for _tj_case in _tj_cases:
            _tj_group = [row for row in _tj_rows if row["case"] == _tj_case]
            _tj_fingerprint = (
                "benchmark/gpu_name", "benchmark/imgsz", "benchmark/batch",
                "benchmark/device", "benchmark/precision", "benchmark/backend",
                "benchmark/torch_version", "benchmark/cuda_runtime",
            )
            _tj_speed_comparable = len({
                tuple(str(row.get(key)) for key in _tj_fingerprint)
                for row in _tj_group
            }) == 1
            _tj_summary = {
                "case": _tj_case, "runs": 3,
                "benchmark/speed_comparable": _tj_speed_comparable,
            }
            for _tj_key in _tj_metric_keys.values():
                if _tj_key.startswith("test_speed/") and not _tj_speed_comparable:
                    continue
                _tj_values = [float(row[_tj_key]) for row in _tj_group]
                _tj_summary[_tj_key + "/mean"] = _tj_statistics.fmean(_tj_values)
                _tj_summary[_tj_key + "/std"] = _tj_statistics.stdev(_tj_values)
            _tj_aggregate.append(_tj_summary)
        with (_tj_project / "summary_aggregate.csv").open("w", newline="", encoding="utf-8") as _tj_stream:
            _tj_fields = sorted(set().union(*(set(row) for row in _tj_aggregate)))
            _tj_writer = _tj_csv.DictWriter(_tj_stream, fieldnames=_tj_fields)
            _tj_writer.writeheader()
            _tj_writer.writerows(_tj_aggregate)
        print("[TINY JOURNAL] All 9 runs verified; aggregate complete.", flush=True)

        # Automatically publish completed artifacts into a NEW namespace only.
        # Never overwrite or silently repair existing remote results.
        _tj_archive = _TJArchive(_tj_repo, token=HF_TOKEN, cache_dir=_tj_cache)
        _tj_uploaded = []
        for _tj_case in _tj_cases:
            for _tj_seed in _tj_seeds:
                _tj_run = _tj_seed_run(_tj_case, _tj_seed)
                _tj_remote_dir = f"{_tj_namespace}/{_tj_case}/seed_{_tj_seed}"
                try:
                    _tj_remote_files = set(_tj_archive._get_api().list_repo_files(
                        repo_id=_tj_repo, repo_type="dataset",
                    ))
                    if any(name.startswith(_tj_remote_dir + "/") for name in _tj_remote_files):
                        # Existing archive is NOT assumed complete without marker and hash check.
                        _tj_marker_remote = _tj_remote_dir + "/upload_complete.json"
                        if _tj_marker_remote not in _tj_remote_files:
                            raise RuntimeError("BLOCKED_REMOTE_EXISTS_WITHOUT_COMPLETION_MARKER")
                        _tj_marker_local = _TJPath(_tj_hf_download(
                            repo_id=_tj_repo, repo_type="dataset", filename=_tj_marker_remote,
                            token=HF_TOKEN, local_dir=str(_tj_cache),
                        ))
                        _tj_remote_marker = _tj_read(_tj_marker_local)
                        _tj_expected = {
                            "weights/best.pt": _tj_sha(_tj_run / "weights/best.pt"),
                            "evaluation_metrics_completion_v1.json": _tj_sha(
                                _tj_run / "evaluation_metrics_completion_v1.json"
                            ),
                            "completion_source.json": _tj_sha(_tj_run / "completion_source.json"),
                        }
                        if not all(_tj_remote_marker.get("artifact_sha256", {}).get(name) == sha
                                   for name, sha in _tj_expected.items()):
                            raise RuntimeError("BLOCKED_REMOTE_ARCHIVE_CONFLICT")
                        _tj_uploaded.append({"case": _tj_case, "seed": _tj_seed, "status": "ALREADY_PUBLISHED"})
                        print(f"[TINY JOURNAL] ALREADY_PUBLISHED {_tj_case}/seed_{_tj_seed}", flush=True)
                        continue
                    _tj_stage = _tj_project / "_publish_stage" / _tj_case / f"seed_{_tj_seed}"
                    _tj_stage.mkdir(parents=True, exist_ok=True)
                    (_tj_stage / "weights").mkdir(exist_ok=True)
                    for _tj_relative in (
                        "weights/best.pt", "recovery_source.json",
                        "evaluation_metrics_completion_v1.json", "completion_source.json",
                    ):
                        _tj_from = _tj_run / _tj_relative
                        _tj_to = _tj_stage / _tj_relative
                        if _tj_to.is_file():
                            if _tj_sha(_tj_to) != _tj_sha(_tj_from):
                                raise RuntimeError("BLOCKED_PUBLISH_STAGE_COLLISION")
                        else:
                            _tj_shutil.copy2(_tj_from, _tj_to)
                    _tj_archive.publish_run(
                        dataset_slug=_tj_namespace, variant=_tj_case, seed=_tj_seed,
                        run_name=f"seed_{_tj_seed}", run_dir=_tj_stage,
                        required_artifacts=(
                            "weights/best.pt", "recovery_source.json",
                            "evaluation_metrics_completion_v1.json", "completion_source.json",
                        ),
                        summary_metadata={
                            "source": _tj_read(_tj_run / "recovery_source.json"),
                            "training": "never_retrained",
                            "evaluation": "original-image merged TinyBenchmark mAP50:95",
                        },
                    )
                    _tj_uploaded.append({"case": _tj_case, "seed": _tj_seed, "status": "PUBLISHED"})
                    print(f"[TINY JOURNAL] PUBLISHED {_tj_case}/seed_{_tj_seed}", flush=True)
                except Exception as _tj_exc:
                    _tj_uploaded.append({
                        "case": _tj_case, "seed": _tj_seed, "status": "BLOCKED_PUBLISH",
                        "reason": f"{type(_tj_exc).__name__}: {str(_tj_exc).replace(HF_TOKEN, '<HF_TOKEN>')}",
                    })
                    print(f"[TINY JOURNAL] BLOCKED_PUBLISH {_tj_case}/seed_{_tj_seed}", flush=True)
                _tj_status["publish_results"] = _tj_uploaded
                _tj_write(_tj_progress_path, _tj_status)
        _tj_status["publish_results"] = _tj_uploaded
        _tj_status["final"] = (
            "COMPLETE" if all(row["status"] in {"PUBLISHED", "ALREADY_PUBLISHED"} for row in _tj_uploaded)
            else "EVALUATED_BUT_PUBLISH_PARTIAL"
        )
        _tj_write(_tj_progress_path, _tj_status)
        print(f"[TINY JOURNAL] {_tj_status['final']}: {_tj_project}", flush=True)


In [ ]:
# Archived cross-YOLO launcher: disabled during FTSC journal Phase 2.
if False:
    # Overnight full pipeline: Git sync -> dataset -> preflight -> 18 runs -> HF -> aggregate.
    # Start the notebook once (Run All). This cell launches the entire suite automatically.
    # On a subsequent Run All, completed HF-backed runs are restored and verified;
    # partial last.pt runs resume without manually switching phases.
    from pathlib import Path as _TpCYPath
    import os as _TpCYOS
    import subprocess as _TpCYSubprocess
    import sys as _TpCYSys
    
    _TpCYRoot = _TpCYPath(yolo_code_dir).resolve()
    _TpCYScript = _TpCYRoot / "utils/ftsc_cross_yolo_overnight_v1.py"
    if not _TpCYScript.is_file():
        raise FileNotFoundError(
            f"Missing overnight runner in GitHub-synced codebase: {_TpCYScript}. "
            "Push the cross-YOLO codebase changes and rerun Git sync."
        )
    _TpCYCommand = [
        _TpCYSys.executable, "-u", str(_TpCYScript),
        "--dataset", "tinyperson",
        "--data-root", str(_TpCYPath(tinyperson_data_root).resolve()),
        "--dataset-root", str(_TpCYPath(tinyperson_prepared_root).resolve()),
        "--device", "cuda",
        "--workers", "4",
    ]
    print("[tinyperson] Starting full cross-YOLO overnight suite (18 runs, auto resume and HF upload)", flush=True)
    _TpCYSubprocess.run(
        _TpCYCommand,
        cwd=str(_TpCYRoot),
        env={**_TpCYOS.environ, "PYTHONUNBUFFERED": "1"},
        check=True,
    )
    print("[tinyperson] Finished all cross-YOLO runs, archives and summaries.", flush=True)


In [ ]:
# FTSC JOURNAL PHASE 2 — unattended Run All (no manual phase edits).
# Uses the frozen Phase-1 dataset runner; no additional YOLO portability suite.
# HF failure/provenance mismatch aborts BEFORE authorizing a new training run.
from pathlib import Path as _FTSCPhase2Path
import os as _ftsc_phase2_os
import subprocess as _ftsc_phase2_subprocess
import sys as _ftsc_phase2_sys

_ftsc_phase2_root = _FTSCPhase2Path(yolo_code_dir).resolve()
_ftsc_phase2_runner = _ftsc_phase2_root / 'utils/ftsc_journal_phase2_one_shot.py'
if not _ftsc_phase2_runner.is_file():
    raise FileNotFoundError(f'Missing synced one-shot paper runner: {_ftsc_phase2_runner}. Push codebase first.')
_ftsc_phase2_data_root = (_FTSCPhase2Path(tinyperson_data_root)).resolve()
_ftsc_phase2_prepared_root = (_FTSCPhase2Path(tinyperson_prepared_root)).resolve()
_ftsc_phase2_pretrained = (_FTSCPhase2Path(yolo_code_dir) / "yolov8n.pt").resolve()
if not _ftsc_phase2_data_root.is_dir():
    raise FileNotFoundError(f'Dataset not extracted before Phase 2: {_ftsc_phase2_data_root}')
_ftsc_phase2_prepared_root.mkdir(parents=True, exist_ok=True)
_ftsc_phase2_command = [
    _ftsc_phase2_sys.executable, '-u', str(_ftsc_phase2_runner),
    '--dataset', 'tinyperson',
    '--data-root', str(_ftsc_phase2_data_root),
    '--dataset-root', str(_ftsc_phase2_prepared_root),
    '--pretrained', str(_ftsc_phase2_pretrained),
    '--device', 'cuda',
    '--workers', '4',
]
print('[tinyperson] FTSC journal one-shot: HF inspect -> verify -> restore/evaluate/resume/train -> publish -> aggregate', flush=True)
_ftsc_phase2_subprocess.run(
    _ftsc_phase2_command,
    cwd=str(_ftsc_phase2_root),
    env={**_ftsc_phase2_os.environ, 'PYTHONUNBUFFERED': '1',
         'PYTHONPATH': str(_ftsc_phase2_root) + _ftsc_phase2_os.pathsep
           + str(_ftsc_phase2_root / 'models_related/ultralytics')},
    check=True,
)
print('[tinyperson] FTSC journal phase2 finished; see runs/tinyperson_ftsc_journal_phase2_v1/phase2_progress.json', flush=True)
